# 02 · Sample a balanced subset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/02_sample.ipynb)

Turn the whole pool into the ~40 items you will actually annotate and study.

```
  01_build_pool_<track>  →▶ 02_sample  →  03_annotate  →  04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/pools/<track>_pool.json` (from 01) |
| **Writes** | `data/gold/<track>_<group>_sample.json` |

---

### Working as a group

These notebooks live in a shared Drive folder, so **all of you can edit at once** — Colab syncs edits like a Google Doc. Two things do *not* work that way:

- **Runtimes are per-person.** Seeing `sampled` in a saved output does not mean `sampled` exists in *your* session. Whoever runs the cells is the **driver**.
- **Files are last-write-wins.** `data/`, `prompts/` and `outputs/` are ordinary files, not Google Docs. Let the driver be the only one running cells that write them.

The **annotation Sheet in notebook 03 is the exception** — that is a real Google Sheet, so annotate it together, all at once.

## Setup — run this first

In Colab, uncomment **one** of the two clone blocks below before running. Colab starts with only this one file; the clone fetches everything *around* it (`scripts/`, `config.py`, `data/`) so the paths resolve.

**Do Option A once, as a group** — then always open the copy in Drive (*File ▸ Open ▸ Drive ▸ `lda2-final-template/notebooks/...`*). Your prompts, gold set and outputs then survive the runtime resetting, and everyone sees the same files.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first.
# ------------------------------------------------------------------
# In Google Colab, UNCOMMENT one of the two blocks below, then run the cell.

# --- Colab Option A: clone into your Google Drive (persists; do this once) ---
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive
# ![ -d lda2-final-template ] || git clone https://github.com/egumasa/lda2-final-template.git lda2-final-template
# %cd /content/drive/MyDrive/lda2-final-template/notebooks

# --- Colab Option B: quick, throwaway clone (changes lost on reset) ---
# !git clone https://github.com/egumasa/lda2-final-template.git
# %cd lda2-final-template/notebooks

# Put scripts/ and config.py on the import path. Works locally AND in Colab
# after the %cd above, because notebooks/ sits beside both.
import sys
sys.path.append("../scripts")
sys.path.append("..")

from config import *      # TRACK, GROUP, SEED, N_PER_CLASS, and every path

from pipeline import *      # load_gold, sample_pool, label_set, save_json, ...

describe()                  # what this notebook is working on


> **Everything above comes from `config.py`** — one file at the top of the repo, which you edit once as a group. That is deliberate: the seed in notebook 02 has to be the seed in notebook 03, and five copies of a number in five notebooks is five chances for them to disagree. If the line it just printed is not your track, your group and your seed, fix `config.py` and re-run this cell.

## What this notebook is for

A **pool** is everything the corpus has, with its natural imbalance. A **sample** is the balanced subset you actually study — equal items per label, so precision, recall, F1 and the confusion matrix all stay readable. Rare labels simply yield fewer; that is a property of the data, and it belongs in your limitations.

Keeping the two separate is also what leaves unused items available as few-shot examples in notebook 04, without showing the model the answers you are testing it on. That is why you sample rather than just taking the first 40 rows.

> **Do not have a pool yet?** Run `01_build_pool_<track>.ipynb` first. To see this notebook work before then, point at `DEMO_POOL_PATH` in the cell below — but the demo pools are small enough that a real sample would eat most of them, and `sample_pool` will warn you when it does.

In [ ]:
# ══ STEP 1 · Load the pool ════════════════════════════════════════════════
# Goal      : open what notebook 01 wrote, and see its natural imbalance.
# Available : load_gold(path)  ->  a list of {id, text, label}
#             POOL_PATH · DEMO_POOL_PATH   (both come from config.py)
# Pointer   : Day 2 S5 step F · Day 3 setup — the same call.
# Produce   : pool      ← later cells use these names
# Try       : from collections import Counter
#             Counter(item["label"] for item in pool)

# ✏️ your code here


### Before you sample — decide `N_PER_CLASS`

You wrote a `MIN_PER_CLASS` at the end of notebook 01: the size of your smallest class. That is a hard ceiling. Ask for more and `sample_pool` gives you everything the rare class has and moves on — your sample is then *not* balanced, and it will not say so twice.

The other ceiling is time. Every item is one API call in notebook 04, at a few seconds each, times the number of rounds — and every item is also a row two of you have to annotate by hand in notebook 03. Around 40 items total is the size this project is built for.

Set `N_PER_CLASS` in `config.py` and re-run the setup cell if you change it.

In [ ]:
# ══ STEP 2 · Draw the balanced sample ═════════════════════════════════════
# Goal      : turn the big pool into ~40 items, balanced across your labels.
# Available : sample_pool(pool, n_per_class, seed)  ->  sampled
#             label_set(items)  ->  the sorted list of labels present
# Pointer   : Day 4 Part A — sample_pool does those four steps in one call.
# Produce   : sampled · LABELS      ← later cells use these names
# Careful   : pass SEED explicitly. A sample you cannot redraw is not a
#             sample anyone can check — and your report has to state it.
# Note      : read the per-label counts it prints. If a class came back
#             short, that is your rare class hitting its ceiling.
# Note      : if it WARNS that you took most of the pool, you are almost
#             certainly still pointed at DEMO_POOL_PATH.

# ✏️ your code here


### Sanity-check what you drew

Two questions worth answering before you commit forty hand-annotations to it:

1. **Is it actually balanced?** Count the labels in `sampled`.
2. **Is there pool left over?** `build_fewshot` in notebook 04 draws its examples from the items you did *not* sample. If the sample is most of the pool, there is nothing uncontaminated left to draw from.

In [ ]:
# ══ STEP 3 · Check the draw ═══════════════════════════════════════════════
# Goal      : confirm the balance, and confirm there is pool left over.
# Available : len(pool) · len(sampled)  ·  Counter(item["label"] for item in sampled)
# Pointer   : Day 4 Part A — the same counts you printed there.
# Produce   : nothing to name — this is a check, not a stage      ← later cells use these names
# Ask       : how many items per label did you get, and does the shortfall
#             match the rare class you spotted in notebook 01?

# ✏️ your code here


## Save it — this is the handoff

The cell below writes the sample to a file. Notebook 03 opens that file to build your annotation sheet, and again at the end to compare your labels against the published ones — so it has to be the same forty items, not a redraw. Even with a fixed seed, save it: a seed reproduces a draw only as long as nobody edits the pool underneath it.

**Next:** open `03_annotate.ipynb`. It starts by loading `data/gold/<track>_<group>_sample.json`.

In [ ]:
save_json(sampled, SAMPLE_PATH, what="sampled items")

# It still carries the PUBLISHED label at this point. Notebook 03 deliberately
# does not copy that into your annotation sheet — you annotate blind — but it
# does use it at the very end, to show you where your group disagreed with the
# corpus. That comparison is one of the more interesting things in your report.
